In [ ]:
pip install snowflake-ml-python -U

In [ ]:
-- VERSION CONFIG
SET FEATURE_VIEW_VERSION = 'V2';
SET MODEL_NAME = 'FRAUD_XGBOOST';
SET MODEL_VERSION = 'V1';
SET SERVICE_NAME = 'FRAUD_DETECTION_SVC';
SET COMPUTE_POOL_NAME = 'ML_ONLINE_CPU_POOL';
-- ROLES
SET PRODUCER_ROLE = 'ACCOUNTADMIN';
SET CONSUMER_ROLE = 'ACCOUNTADMIN';

In [ ]:
USE ROLE SFLK_GBU_A01A0E_SUPPORTL3_DEV;
CREATE SCHEMA IF NOT EXISTS A01A0E_GBU_FINCRIME_POC.FRAUD_ML_SERVICE_FS;
USE DATABASE A01A0E_GBU_FINCRIME_POC;
USE SCHEMA FRAUD_ML_SERVICE_FS;

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n_transactions = 200

data = pd.DataFrame({
    "TRANSACTION_ID": [f"TXN_{i:04d}" for i in range(n_transactions)],
    "TRANSACTION_AMOUNT": np.round(np.random.uniform(5, 5000, n_transactions), 2),
    "MERCHANT_CATEGORY": np.random.choice([0, 1, 2, 3, 4], n_transactions),
    "DISTANCE_FROM_HOME": np.round(np.random.uniform(0, 500, n_transactions), 2),
    "TIME_SINCE_LAST_TXN": np.round(np.random.uniform(0.1, 72, n_transactions), 2),
    "DAILY_TXN_COUNT": np.random.randint(1, 20, n_transactions),
    "IS_FRAUD": np.random.choice([0, 1], n_transactions, p=[0.92, 0.08]),
})

print(f"Dataset shape: {data.shape}")
data.head()

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity

session = get_active_session()
session.use_database("A01A0E_GBU_FINCRIME_POC")
session.use_schema("FRAUD_ML_SERVICE_FS")

session.create_dataframe(data).write.save_as_table("TRANSACTION_FEATURES", mode="overwrite")
print("Table TRANSACTION_FEATURES created with", session.table("TRANSACTION_FEATURES").count(), "rows")

In [ ]:
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode
from snowflake.ml.feature_store import OnlineConfig, OnlineStoreType

fs = FeatureStore(
    session=session,
    database="A01A0E_GBU_FINCRIME_POC",
    name="FRAUD_ML_SERVICE_FS",
    default_warehouse="GBU_A01A0E_FINCRIME_XS_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

transaction_entity = Entity(name="TRANSACTION", join_keys=["TRANSACTION_ID"])
fs.register_entity(transaction_entity)
print("Entity registered:", transaction_entity.name)

In [ ]:
import time

PRODUCER_ROLE = "SFLK_GBU_A01A0E_SER_READWRITE_DEV"
CONSUMER_ROLE = "SFLK_GBU_A01A0E_SER_READWRITE_DEV"

try:
    result = fs.create_online_service(
        producer_role=PRODUCER_ROLE,
        consumer_role=CONSUMER_ROLE
    )
    print(f"Create online service: {result}")
except Exception as e:
    print(f"Online service already exists or error: {e}")

# Poll until RUNNING
for i in range(60):
    status = fs.get_online_service_status()
    current_status = status.status if hasattr(status, 'status') else str(status)
    print(f"[{i+1}] Online service status: {current_status}")
    if current_status == "RUNNING":
        print("Online service is ready!")
        break
    time.sleep(10)
else:
    raise TimeoutError("Service did not reach RUNNING within 10 min.")

In [ ]:
FEATURE_VIEW_VERSION = "v1"

source_df = session.table("A01A0E_GBU_FINCRIME_POC.FRAUD_ML_SERVICE_FS.TRANSACTION_FEATURES").select(
    "TRANSACTION_ID", "TRANSACTION_AMOUNT", "MERCHANT_CATEGORY",
    "DISTANCE_FROM_HOME", "TIME_SINCE_LAST_TXN", "DAILY_TXN_COUNT"
)

txn_fv = FeatureView(
    name="TRANSACTION_FRAUD_FEATURES",
    entities=[transaction_entity],
    feature_df=source_df,
    desc="Transaction fraud detection features",
    refresh_freq="1 minute",
    online_config=OnlineConfig(
        enable=True,
        target_lag="10s",
        store_type=OnlineStoreType.POSTGRES,
    ),
)

txn_fv = fs.register_feature_view(feature_view=txn_fv, version=FEATURE_VIEW_VERSION, overwrite=True)
print(f"Feature View registered: {txn_fv.name} version {txn_fv.version}")
print(f"Online store type: POSTGRES")

In [ ]:
import os

os.environ['SNOWFLAKE_PAT'] = session.connection.rest.token

In [ ]:
from snowflake.ml.feature_store.feature_view import StoreType

os.environ['SNOWFLAKE_PAT'] = session.connection.rest.token

txn_fv = fs.get_feature_view("TRANSACTION_FRAUD_FEATURES", FEATURE_VIEW_VERSION)
print(f"Store type: {txn_fv.online_config.store_type}")
print(f"Target lag: {txn_fv.online_config.target_lag}")

test_keys = [["TXN_0001"], ["TXN_0010"], ["TXN_0050"]]

try:
    online_result = fs.read_feature_view(
        txn_fv,
        keys=test_keys,
        store_type=StoreType.ONLINE
    )
    print("\nOnline (Postgres) feature retrieval successful!")
except Exception as e:
    print(f"\nOnline store unreachable ({type(e).__name__}), falling back to offline store...")
    online_result = fs.read_feature_view(
        txn_fv,
        keys=test_keys,
        store_type=StoreType.OFFLINE
    )
    print("Offline feature retrieval successful (fallback).")

online_result

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

FEATURES = ["TRANSACTION_AMOUNT", "MERCHANT_CATEGORY", "DISTANCE_FROM_HOME", 
    "TIME_SINCE_LAST_TXN", "DAILY_TXN_COUNT"]
TARGET = "IS_FRAUD"

X = data[FEATURES]
y = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBClassifier(
    n_estimators=50,
    max_depth=4,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"ROC AUC:  {roc_auc_score(y_test, y_proba):.3f}")


In [ ]:
from snowflake.ml.registry import Registry

MODEL_NAME = "FRAUD_DETECTION_MODEL"
MODEL_VERSION = "v1"

reg = Registry(session=session, database_name="A01A0E_GBU_FINCRIME_POC", schema_name="FRAUD_ML_SERVICE_FS")

try:
    mv = reg.get_model(MODEL_NAME).version(MODEL_VERSION)
    print(f"Using existing model: {MODEL_NAME}/{MODEL_VERSION}")
except Exception:
    sample_input = X_train.head(5)
    mv = reg.log_model(
        model_name=MODEL_NAME,
        version_name=MODEL_VERSION,
        model=model,
        sample_input_data=sample_input,
        conda_dependencies=["xgboost"],
    )
    print(f"Model registered: {MODEL_NAME} version {MODEL_VERSION}")

funcs = mv.show_functions()
print(f"Functions: {[f['name'] if isinstance(f, dict) else str(f) for f in funcs]}")

In [ ]:
-- Using existing compute pool (CREATE COMPUTE POOL requires account-level privileges)
DESCRIBE COMPUTE POOL SYSTEM_COMPUTE_POOL_CPU;

In [ ]:
SERVICE_NAME = "FRAUD_DETECTION_SERVICE"
COMPUTE_POOL_NAME = "SYSTEM_COMPUTE_POOL_CPU"

mv.create_service(
    service_name=SERVICE_NAME,
    service_compute_pool=COMPUTE_POOL_NAME,
    ingress_enabled=True,
    max_instances=1,
)
print(f"Service deployment initiated: {SERVICE_NAME}")

In [ ]:
services_df = mv.list_services()
print("Active services:")
print(services_df.to_string())